In [11]:
import requests
import io
import pandas as pd

In [24]:
def extraer_tablas_investigacion(nombre_estrella):
    """
    Extrae parámetros físicos avanzados de la estrella (incluyendo tipo espectral)
    y del planeta (masas reales y masas mínimas Mp sin i en Tierras y Júpiter).
    Incluye sanitización para nombres de estrellas con apóstrofes.
    """
    url_base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    
    # CORRECCIÓN CLAVE: Sanitizamos el string para SQL duplicando la comilla simple
    nombre_estrella_sql = nombre_estrella.replace("'", "''")
    
    query = f"""
    SELECT 
        hostname AS estrella,
        st_spectype AS tipo_espectral,
        st_teff AS temp_estrella_k,
        st_rad AS radio_estrella_sol,
        st_mass AS masa_estrella_sol,
        st_teff_reflink AS ref_estrella,
        pl_name AS nombre_planeta,
        pl_orbper AS periodo_dias,
        pl_rade AS radio_tierra,
        pl_bmasse AS masa_tierra,
        pl_bmassj AS masa_jupiter,
        pl_msinie AS mpsini_tierra,
        pl_msinij AS mpsini_jupiter,
        discoverymethod AS metodo_descubrimiento,
        pl_orbper_reflink AS ref_planeta
    FROM pscomppars
    WHERE hostname = '{nombre_estrella_sql}'
    """
    
    parametros = {
        "query": query,
        "format": "csv"
    }
    
    print(f"Buscando el sistema {nombre_estrella} en la base de datos...\n")
    respuesta = requests.get(url_base, params=parametros)
    
    if respuesta.status_code == 200:
        datos_csv = io.StringIO(respuesta.text)
        df_completo = pd.read_csv(datos_csv)
        
        if df_completo.empty:
            print(f"No se encontró información para '{nombre_estrella}'.")
            return None, None
            
        # 1. Tabla de la Estrella
        columnas_estrella = ['estrella', 'tipo_espectral', 'temp_estrella_k', 'radio_estrella_sol', 'masa_estrella_sol', 'ref_estrella']
        df_estrella = df_completo[columnas_estrella].drop_duplicates().reset_index(drop=True)
        df_estrella.columns = ['Estrella', 'Tipo_Espectral', 'Temp_K', 'Radio_Sol', 'Masa_Sol', 'Referencia_Estrella']
        
        # 2. Tabla de los Planetas
        columnas_planetas = ['nombre_planeta', 'periodo_dias', 'radio_tierra', 
                             'masa_tierra', 'masa_jupiter', 'mpsini_tierra', 'mpsini_jupiter', 
                             'metodo_descubrimiento', 'ref_planeta']
        df_planetas = df_completo[columnas_planetas].reset_index(drop=True)
        df_planetas.columns = ['Planeta', 'Periodo(Dias)', 'Radio(Tierra)', 
                               'Masa(Tierra)', 'Masa(Jupiter)', 'Mp_sin_i(Tierra)', 'Mp_sin_i(Jupiter)', 
                               'Metodo', 'Referencia_Planeta']
        
        return df_estrella, df_planetas
    else:
        print(f"Error HTTP {respuesta.status_code}: {respuesta.text}")
        return None, None

In [26]:
# ==========================================
# EJECUCIÓN Y EXPORTACIÓN A LATEX (OVERLEAF)
# ==========================================
if __name__ == "__main__":
    
    estrella_objetivo = "Barnard's star" 
    
    df_star, df_planets = extraer_tablas_investigacion(estrella_objetivo)
    
    if df_star is not None and df_planets is not None:
        
        # Opcional: Reemplazar guiones bajos en los nombres de columnas 
        # para que LaTeX no los interprete como subíndices matemáticos si están fuera de modo matemático ($).
        df_star.columns = [col.replace('_', ' ') for col in df_star.columns]
        df_planets.columns = [col.replace('_', ' ') for col in df_planets.columns]

        # Nombres de los archivos físicos que se generarán
        archivo_estrella = "tabla_estrella.tex"
        archivo_planetas = "tabla_planetas.tex"
        
        # Intentamos generar el código LaTeX (manejando la sintaxis moderna de Pandas 2.0+)
        try:
            # Pandas 2.0+ utiliza el atributo 'style' para la exportación a LaTeX
            latex_star = df_star.style.format(na_rep='--').to_latex()
            latex_planets = df_planets.style.format(na_rep='--').to_latex()
        except AttributeError:
            # Soporte para versiones anteriores de Pandas (1.x)
            latex_star = df_star.to_latex(index=False, na_rep='--')
            latex_planets = df_planets.to_latex(index=False, na_rep='--')
            
        # 1. GUARDAR EN ARCHIVOS FÍSICOS
        with open(archivo_estrella, 'w', encoding='utf-8') as f:
            f.write("% Generado automáticamente desde el script de Exoplanet Archive\n")
            f.write(latex_star)
            
        with open(archivo_planetas, 'w', encoding='utf-8') as f:
            f.write("% Generado automáticamente desde el script de Exoplanet Archive\n")
            f.write(latex_planets)
            
        print(f"¡Éxito! Las tablas de '{estrella_objetivo}' se guardaron en los archivos '{archivo_estrella}' y '{archivo_planetas}'.")
        
        # 2. IMPRIMIR EN CONSOLA PARA COPIAR Y PEGAR
        print("\n" + "="*80)
        print("CÓDIGO LATEX PARA OVERLEAF: ESTRELLA ANFITRIONA")
        print("="*80)
        print(latex_star)
        
        print("\n" + "="*80)
        print("CÓDIGO LATEX PARA OVERLEAF: EXOPLANETAS")
        print("="*80)
        print(latex_planets)

Buscando el sistema Barnard's star en la base de datos...

¡Éxito! Las tablas de 'Barnard's star' se guardaron en los archivos 'tabla_estrella.tex' y 'tabla_planetas.tex'.

CÓDIGO LATEX PARA OVERLEAF: ESTRELLA ANFITRIONA
\begin{tabular}{lllrrrl}
 & Estrella & Tipo Espectral & Temp K & Radio Sol & Masa Sol & Referencia Estrella \\
0 & Barnard's star & M3.5-4 V & 3195.000000 & 0.185000 & 0.162000 & <a refstr=GONZALEZ_HERNANDEZ_ET_AL_2024 href=https://ui.adsabs.harvard.edu/abs/2024A&A...690A..79G/abstract target=ref>Gonz&aacute;lez Hern&aacute;ndez et al. 2024</a> \\
\end{tabular}


CÓDIGO LATEX PARA OVERLEAF: EXOPLANETAS
\begin{tabular}{llrrrrrrll}
 & Planeta & Periodo(Dias) & Radio(Tierra) & Masa(Tierra) & Masa(Jupiter) & Mp sin i(Tierra) & Mp sin i(Jupiter) & Metodo & Referencia Planeta \\
0 & Barnard d & 2.340200 & 0.694000 & 0.263000 & 0.000827 & 0.263000 & 0.000827 & Radial Velocity & <a refstr=BASANT_ET_AL__2025 href=https://ui.adsabs.harvard.edu/abs/2025ApJ...982L...1B/abstract ta